# Sample Prediction - `claim_denial_model`

**What this notebook does**

1. Resolves the registered model dependency spec so a fresh Databricks kernel can install the exact packages bundled with the current `champion` model.
2. Loads the gate-passing model from the MLflow Registry alias `healthcare.ml.claim_denial_model@champion` (with a local-pickle fallback for offline runs).
3. Pulls a small sample of claims from `healthcare.gold.claim_features` (with a synthetic in-memory fallback so the notebook still runs locally).
4. Calls `predict_single` on a single claim and `predict_batch` across the sample.
5. Prints the WEEK4-shaped output (`Claim ID: ... / Risk: HIGH (0.82)`) plus a quick risk-tier distribution.

## 1. Bootstrap model dependencies

On a fresh Databricks kernel, resolve the dependency file bundled with the current `champion` model and install from that file before loading the registry model. This keeps the notebook aligned with future dependency changes without editing package names here.


In [0]:
from src.ml.predict import get_registry_model_dependencies

In [0]:
REGISTRY_NAME = "healthcare.ml.claim_denial_model"
REGISTRY_ALIAS = "champion"
dependency_spec = get_registry_model_dependencies(
    REGISTRY_NAME, REGISTRY_ALIAS)
print(f"Dependency spec: {dependency_spec}")
print("On a fresh Databricks kernel run:")
print(f"%pip install -q -r {dependency_spec}")
print("dbutils.library.restartPython()")

In [0]:
# Databricks-only bootstrap cell. Run this once on a fresh kernel.
%pip install -q -r {dependency_spec}
dbutils.library.restartPython()


In [0]:
from __future__ import annotations
from src.ml.predict import (
    RiskLevel,
    load_from_registry,
    load_trained_model,
    predict_batch,
    predict_single,
)
from src.ml import FEATURE_COLUMNS

import sys
from pathlib import Path

import pandas as pd


def _find_project_root(start: Path) -> Path:
    """Walk up from `start` until a pyproject.toml is found (project root marker)."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


print(f"Project root: {PROJECT_ROOT}")

## 2. Load the model

After the dependency bootstrap is complete, try the MLflow Registry first (production path). Fall back to the local pickle written by `scripts/train_denial_model.py` so the notebook still runs on a laptop where the registry is not reachable. If both fail, the surfaced exception message tells you which one to fix.


In [0]:
REGISTRY_NAME = "healthcare.ml.claim_denial_model"
REGISTRY_ALIAS = "champion"
PICKLE_FALLBACK = PROJECT_ROOT / "models" / "claim_denial_model.pkl"

model = None
load_source = None
registry_error = None
try:
    model = load_from_registry(REGISTRY_NAME, REGISTRY_ALIAS)
    load_source = f"models:/{REGISTRY_NAME}@{REGISTRY_ALIAS}"
except Exception as exc:  # pragma: no cover - notebook UX
    registry_error = f"{exc.__class__.__name__}: {exc}"
    print(f"Registry load failed: {registry_error}")
    print(f"Falling back to pickle at {PICKLE_FALLBACK}.")
    if not PICKLE_FALLBACK.exists():
        raise FileNotFoundError(
            f"Registry load failed AND no pickle at {PICKLE_FALLBACK}.\n"
            f"  Registry error: {registry_error}\n"
            f"  Fix one of:\n"
            f"    - Install the model-derived dependency spec shown in section 1 and restart Python\n"
            f"    - Train + register: from scripts.train_denial_model import main; main(['--tune'])\n"
            f"    - Verify Unity Catalog schema: CREATE SCHEMA IF NOT EXISTS healthcare.ml;\n"
            f"    - Verify the alias is set on a registered version of the model."
        ) from exc
    model = load_trained_model(PICKLE_FALLBACK)
    load_source = str(PICKLE_FALLBACK)

print(f"Loaded {type(model).__name__} from: {load_source}")

## 3. Pull a sample of claims

Prefer the Gold feature table (real engineered features). Fall back to a small in-memory sample with the same shape as `FEATURE_COLUMNS` so the notebook is self-contained.


In [0]:
GOLD_TABLE = "healthcare.gold.claim_features"
SAMPLE_SIZE = 5

sample_df = None
try:
    from pyspark.sql import SparkSession

    spark = SparkSession.builder.getOrCreate()
    sample_df = spark.table(GOLD_TABLE).limit(SAMPLE_SIZE).toPandas()
    sample_source = GOLD_TABLE
except Exception as exc:  # pragma: no cover - notebook UX
    print(
        f"Spark/Gold unavailable ({exc.__class__.__name__}: {exc}); using synthetic sample.")
    sample_df = pd.DataFrame(
        {
            "claim_id": [f"DEMO{i:03d}" for i in range(SAMPLE_SIZE)],
            "is_procedure_missing": [False, True, False, False, False],
            "is_amount_missing": [False, False, True, False, False],
            "amount_to_benchmark_ratio": [1.05, 0.0, 0.0, 3.10, 1.20],
            "billed_vs_avg_cost": [1.05, 0.0, 0.0, 2.80, 1.15],
            "high_cost_flag": [False, False, False, True, False],
            "severity_procedure_mismatch": [False, False, False, True, False],
            "specialty_diagnosis_mismatch": [False, False, False, False, True],
            "provider_location_missing": [False, False, False, False, False],
            "diagnosis_severity_encoded": [0, 1, 1, 1, 0],
            "diagnosis_count": [4, 6, 3, 8, 5],
            "provider_claim_count": [25, 40, 15, 60, 30],
            "provider_claim_count_30d": [3, 5, 1, 8, 4],
            "provider_risk_score": [0.10, 0.55, 0.20, 0.78, 0.35],
        }
    )
    sample_source = "in-memory synthetic"

print(f"Loaded {len(sample_df)} sample claims from: {sample_source}")
sample_df.head()

## 4. Single-claim prediction


In [0]:
first = sample_df.iloc[0]
feature_dict = {col: first[col]
                for col in FEATURE_COLUMNS if col in first.index}
result = predict_single(model, feature_dict)

claim_id = first["claim_id"] if "claim_id" in first.index else "<unknown>"
print(f"Claim ID: {claim_id}")
print(f"Risk: {result['risk_level']} ({result['denial_probability']:.2f})")

## 5. Batch prediction across the sample


In [0]:
scored = predict_batch(model, sample_df)
scored

## 6. Risk-tier distribution

On a healthy production sample expect a mix dominated by `LOW`/`MEDIUM` with a smaller `HIGH` cohort — those are the claims the remediation tool should triage first.


In [0]:
scored["risk_level"].value_counts().reindex(
    [level.value for level in RiskLevel], fill_value=0)